# AIML-01 Fraud Detection: Day 3-4, Features + Feast

**Goal (portal step 2):** engineer features such as transaction velocity, amount versus the account's average, and store them in **Feast** so training and serving use *identical* features.

**What your Day 1-2 numbers told us:** each sender appears once, so "velocity per sender" is impossible. We build the history features on the **receiver** side instead (how busy and how large is the account receiving the money), plus transaction-level features.

**The key idea you should be able to explain:** a feature for a transaction must only use information that existed *before* that transaction. Otherwise the model cheats. Feast gives us "point-in-time" lookups that enforce this, and the same stored values are what the live API will read later.

Run cells top to bottom. If Colab asks you to *restart the runtime* after the install cell, do it, then run again from the top.

## 1. Setup

In [ ]:
!pip install -q feast pyarrow
import os, glob, sys, json
from datetime import timedelta
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.


## 2. Load the data (same as Day 1-2)

In [ ]:
CSV_PATH = os.environ.get("PAYSIM_CSV")   # leave as None to download automatically
if CSV_PATH is None:
    import kagglehub
    folder = kagglehub.dataset_download("moonknightmarvel/paysim")
    CSV_PATH = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)[0]
print("Using:", CSV_PATH)
raw = pd.read_csv(CSV_PATH)
print(f"{len(raw):,} rows")

100%|██████████| 178M/178M [00:01<00:00, 173MB/s]

Extracting files...


Using: /root/.cache/kagglehub/datasets/moonknightmarvel/paysim/versions/1/paysim.csv
6,362,620 rows


## 3. Keep only the transaction types that can be fraud
Check first, do not assume: where does fraud actually occur?

In [ ]:
print(raw.groupby("type")["isFraud"].agg(["count", "sum"]))
fraud_types = sorted(raw.loc[raw.isFraud == 1, "type"].unique())
print("\nTypes containing fraud:", fraud_types)

df = raw[raw["type"].isin(fraud_types)].reset_index(drop=True)
del raw
print(f"\nKept {len(df):,} rows ({df.isFraud.sum():,} fraud, {100*df.isFraud.mean():.3f}%)")

            count   sum
type                   
CASH_IN   1399284     0
CASH_OUT  2237500  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   532909  4097

Types containing fraud: ['CASH_OUT', 'TRANSFER']

Kept 2,770,409 rows (8,213 fraud, 0.296%)


## 4. Build the features
Three groups:

| Group | Features | Notes |
|---|---|---|
| **Receiver history** (from Feast) | `rx_prev_count`, `rx_avg_amount`, `rx_hours_since_last` | Only transactions in *earlier hours* count |
| **Transaction** | `amount`, `hour_of_day`, `type_is_transfer`, `amount_to_rx_avg` | Known at the moment of the transaction |
| **Balance** (flagged) | `amount_to_balance_ratio`, `orig_emptied`, balance errors, raw balances | Strong, but they can leak the answer, so we keep them separate and compare with and without them |

The history window uses `step` (one step = one hour). A transaction only sees receiver history from **strictly earlier steps**, so features never include the transaction itself or same-hour neighbours.

In [ ]:
def add_receiver_history(df):
    """Receiver state BEFORE each transaction's step (no peeking)."""
    d = df[["nameDest", "step", "amount"]].copy()
    d["_i"] = np.arange(len(d))
    d = d.sort_values(["nameDest", "step", "_i"], kind="mergesort")
    g = d.groupby("nameDest", sort=False)
    d["cnt"] = g.cumcount() + 1
    d["sum"] = g["amount"].cumsum()
    # one snapshot per (receiver, step): the state AFTER that step
    snap = d.drop_duplicates(["nameDest", "step"], keep="last")[["nameDest", "step", "cnt", "sum"]].copy()
    sg = snap.groupby("nameDest", sort=False)
    snap["prev_cnt"] = sg["cnt"].shift(1).fillna(0)
    snap["prev_sum"] = sg["sum"].shift(1).fillna(0.0)
    snap["prev_step"] = sg["step"].shift(1)
    out = df.merge(snap[["nameDest", "step", "prev_cnt", "prev_sum", "prev_step"]],
                   on=["nameDest", "step"], how="left")
    return out, snap


def build_features(df):
    df, snap = add_receiver_history(df)
    f = pd.DataFrame(index=df.index)
    # receiver history
    f["rx_prev_count"] = df["prev_cnt"]
    f["rx_avg_amount"] = np.where(df["prev_cnt"] > 0, df["prev_sum"] / df["prev_cnt"].clip(lower=1), 0.0)
    f["rx_hours_since_last"] = np.where(df["prev_step"].notna(), df["step"] - df["prev_step"], -1.0)
    # transaction
    f["amount"] = df["amount"]
    f["hour_of_day"] = df["step"] % 24
    f["type_is_transfer"] = (df["type"] == "TRANSFER").astype(int)
    f["amount_to_rx_avg"] = np.where(f["rx_avg_amount"] > 0, df["amount"] / f["rx_avg_amount"].where(f["rx_avg_amount"] > 0, 1), 0.0)
    # balance group (flagged)
    f["amount_to_balance_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)
    f["orig_emptied"] = ((df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)).astype(int)
    f["orig_balance_error"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
    f["dest_balance_error"] = df["newbalanceDest"] - df["oldbalanceDest"] - df["amount"]
    f["oldbalanceOrg"] = df["oldbalanceOrg"]; f["newbalanceOrig"] = df["newbalanceOrig"]
    f["oldbalanceDest"] = df["oldbalanceDest"]; f["newbalanceDest"] = df["newbalanceDest"]
    # keys and label
    f["step"] = df["step"]; f["nameDest"] = df["nameDest"]; f["isFraud"] = df["isFraud"]
    return f, snap

features, snap = build_features(df)

BASE_FEATURES = ["rx_prev_count", "rx_avg_amount", "rx_hours_since_last",
                 "amount", "hour_of_day", "type_is_transfer", "amount_to_rx_avg"]
BALANCE_FEATURES = ["amount_to_balance_ratio", "orig_emptied", "orig_balance_error", "dest_balance_error",
                    "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
features.head()

,rx_prev_count,rx_avg_amount,rx_hours_since_last,amount,hour_of_day,type_is_transfer,amount_to_rx_avg,amount_to_balance_ratio,orig_emptied,orig_balance_error,dest_balance_error,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,step,nameDest,isFraud
0,0.000,0.000,-1.000,181.000,1,1,0.000,0.995,1,0.000,-181.000,181.000,0.000,0.000,0.000,1,C553264065,1
1,0.000,0.000,-1.000,181.000,1,0,0.000,0.995,1,0.000,"-21,363.000",181.000,0.000,"21,182.000",0.000,1,C38997010,1
2,0.000,0.000,-1.000,"229,133.940",1,0,0.000,14.951,1,"-213,808.940","-182,703.500","15,325.000",0.000,"5,083.000","51,513.440",1,C476402209,0
3,0.000,0.000,-1.000,"215,310.300",1,1,0.000,304.972,1,"-214,605.300","-237,735.300",705.000,0.000,"22,425.000",0.000,1,C1100439041,0
4,0.000,0.000,-1.000,"311,685.890",1,1,0.000,28.764,1,"-300,850.890","2,401,220.000","10,835.000",0.000,"6,267.000","2,719,172.890",1,C932583850,0


## 5. Do the features separate fraud from normal?
Compare the average of each feature for the two classes. We want to see differences, and we want to notice which ones look *too* good.

In [ ]:
cmp = features.groupby("isFraud")[BASE_FEATURES + BALANCE_FEATURES].mean().T
cmp.columns = ["normal", "fraud"]
cmp["ratio (fraud/normal)"] = cmp["fraud"] / cmp["normal"].replace(0, np.nan)
display(cmp)

print("Receivers with prior history:")
print(features.groupby("isFraud")["rx_prev_count"].apply(lambda s: f"{(s > 0).mean():.1%} had earlier transactions"))

,normal,fraud,ratio (fraud/normal)
rx_prev_count,5.226,1.763,0.337
rx_avg_amount,"237,319.975","85,248.967",0.359
rx_hours_since_last,30.697,35.438,1.154
amount,"314,115.496","1,467,967.299",4.673
hour_of_day,15.320,11.546,0.754
type_is_transfer,0.191,0.499,2.606
amount_to_rx_avg,2.274,7.473,3.286
amount_to_balance_ratio,"158,167.398","1,161.967",0.007
orig_emptied,0.427,0.976,2.283
orig_balance_error,"-286,803.510","-10,692.325",0.037


Receivers with prior history:
isFraud
0    80.6% had earlier transactions
1    32.7% had earlier transactions
Name: rx_prev_count, dtype: object


**Write down in your notes:** which base features show a clear difference between fraud and normal? Does fraud go to *new* receivers (low `rx_prev_count`)? That is a pattern an analyst would recognise.

## 6. Save the full feature table
Day 5-6 trains XGBoost on this file.

In [ ]:
features.to_parquet("outputs/features_full.parquet", index=False)
json.dump({"base": BASE_FEATURES, "balance": BALANCE_FEATURES}, open("outputs/feature_groups.json", "w"), indent=2)
print("Saved outputs/features_full.parquet:", features.shape)

Saved outputs/features_full.parquet: (2770409, 18)


## 7. Feast: the feature store
**Receiver history** is what a live service must look up quickly, so it goes in Feast. A transaction's own fields (amount, hour, type) arrive with the request, so the API computes those itself.

Plan:
1. Turn the receiver snapshots into a Feast **feature view** (each row = the receiver's state after a given hour).
2. Ask Feast for **historical** features at the moment before each transaction (training path).
3. Check Feast's answer equals the pandas features above (this proves training and serving consistency).
4. Load the latest state into the **online store** and read it back (serving path).

To keep it fast in Colab, the Feast demo uses all fraud plus a random sample of normal transactions.

In [ ]:
BASE_TS = pd.Timestamp("2026-01-01", tz="UTC")     # step 0; one step = one hour
rng = np.random.default_rng(42)

fraud_idx = features.index[features.isFraud == 1]
normal_idx = rng.choice(features.index[features.isFraud == 0], size=min(20000, int((features.isFraud == 0).sum())), replace=False)
sample = features.loc[np.concatenate([fraud_idx, normal_idx])].copy()
sample_receivers = set(sample["nameDest"])

feast_snap = snap[snap["nameDest"].isin(sample_receivers)].copy()
feast_snap["event_timestamp"] = BASE_TS + pd.to_timedelta(feast_snap["step"], unit="h")
feast_snap = feast_snap.rename(columns={"cnt": "rx_txn_count_total", "sum": "rx_amount_sum_total", "step": "rx_last_step"})
feast_snap = feast_snap[["nameDest", "event_timestamp", "rx_txn_count_total", "rx_amount_sum_total", "rx_last_step"]]
feast_snap["rx_txn_count_total"] = feast_snap["rx_txn_count_total"].astype("int64")
feast_snap["rx_amount_sum_total"] = feast_snap["rx_amount_sum_total"].astype("float64")
feast_snap["rx_last_step"] = feast_snap["rx_last_step"].astype("int64")

os.makedirs("feature_repo/data", exist_ok=True)
feast_snap.to_parquet("feature_repo/data/receiver_snapshots.parquet", index=False)
print(f"{len(sample):,} sample transactions, {len(sample_receivers):,} receivers, {len(feast_snap):,} snapshot rows")

28,213 sample transactions, 27,119 receivers, 236,264 snapshot rows


In [ ]:
%%writefile feature_repo/feature_store.yaml
project: fraud_platform
registry: data/registry.db
provider: local
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3

Writing feature_repo/feature_store.yaml


In [ ]:
%%writefile feature_repo/features.py
from datetime import timedelta
from pathlib import Path
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64, Int64

receiver = Entity(name="receiver", join_keys=["nameDest"], description="Account that receives the money")

receiver_source = FileSource(
    path=str(Path(__file__).parent / "data" / "receiver_snapshots.parquet"),
    timestamp_field="event_timestamp",
)

receiver_stats = FeatureView(
    name="receiver_stats",
    entities=[receiver],
    ttl=timedelta(days=365),
    schema=[
        Field(name="rx_txn_count_total", dtype=Int64),
        Field(name="rx_amount_sum_total", dtype=Float64),
        Field(name="rx_last_step", dtype=Int64),
    ],
    source=receiver_source,
)

Writing feature_repo/features.py


In [ ]:
sys.path.insert(0, "feature_repo")
from feast import FeatureStore
import features as feast_defs

store = FeatureStore(repo_path="feature_repo")
store.apply([feast_defs.receiver, feast_defs.receiver_stats])
print("Registered:", [fv.name for fv in store.list_feature_views()])

/content/feature_repo/features.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'receiver'.
  receiver = Entity(name="receiver", join_keys=["nameDest"], description="Account that receives the money")
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Registered: ['receiver_stats']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 8. Training path: point-in-time lookup
For every sample transaction we ask Feast: "what did you know about this receiver **one second before this hour started**?" Feast returns the latest snapshot at or before that moment.

In [ ]:
entity_df = sample[["nameDest", "step"]].copy()
entity_df["event_timestamp"] = BASE_TS + pd.to_timedelta(entity_df["step"], unit="h") - pd.Timedelta(seconds=1)
entity_keys = entity_df[["nameDest", "event_timestamp"]].drop_duplicates()

hist = store.get_historical_features(
    entity_df=entity_keys,
    features=["receiver_stats:rx_txn_count_total", "receiver_stats:rx_amount_sum_total", "receiver_stats:rx_last_step"],
).to_df()
print(hist.shape)
hist.head()

(18807, 5)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,nameDest,event_timestamp,rx_txn_count_total,rx_amount_sum_total,rx_last_step
0,C1032986144,2026-01-01 01:59:59+00:00,1,"7,206.330",1
1,C2089016471,2026-01-01 06:59:59+00:00,1,"34,115.820",1
2,C2094777811,2026-01-01 03:59:59+00:00,1,"19,772.530",1
3,C824009085,2026-01-01 06:59:59+00:00,3,"141,241.850",1
4,C97730845,2026-01-01 02:59:59+00:00,8,"3,744,204.730",2


## 9. The consistency check (the important one)
Do Feast's values equal the pandas features we used for training? If yes, the live system will see the same numbers the model trained on.

In [ ]:
chk = entity_df.merge(hist, on=["nameDest", "event_timestamp"], how="left")
chk.index = entity_df.index

feast_count = chk["rx_txn_count_total"].fillna(0)
feast_avg = np.where(feast_count > 0, chk["rx_amount_sum_total"].fillna(0) / feast_count.clip(lower=1), 0.0)
feast_since = np.where(chk["rx_last_step"].notna(), chk["step"] - chk["rx_last_step"], -1.0)

pairs = {
    "rx_prev_count": (sample["rx_prev_count"].to_numpy(), feast_count.to_numpy()),
    "rx_avg_amount": (sample["rx_avg_amount"].to_numpy(), feast_avg),
    "rx_hours_since_last": (sample["rx_hours_since_last"].to_numpy(), feast_since),
}
all_ok = True
for name, (a, b) in pairs.items():
    ok = np.allclose(a, b, rtol=1e-6, atol=1e-6)
    all_ok &= ok
    print(f"{name:22s} pandas == Feast on {len(a):,} rows: {ok}")
assert all_ok, "Feast and pandas features differ. Do not continue; send Claude this output."
print("\nTraining/serving consistency check PASSED")

rx_prev_count          pandas == Feast on 28,213 rows: True
rx_avg_amount          pandas == Feast on 28,213 rows: True
rx_hours_since_last    pandas == Feast on 28,213 rows: True

Training/serving consistency check PASSED


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 10. Serving path: the online store
The live API will not read files. It reads the **online store**, which holds the latest state per receiver. Load it and look one up, exactly as the API will.

In [ ]:
store.materialize(
    start_date=feast_snap["event_timestamp"].min().to_pydatetime() - timedelta(days=1),
    end_date=feast_snap["event_timestamp"].max().to_pydatetime() + timedelta(days=1),
)
demo_receivers = sample.loc[sample["rx_prev_count"] > 0, "nameDest"].head(3).tolist() or sample["nameDest"].head(3).tolist()
online = store.get_online_features(
    features=["receiver_stats:rx_txn_count_total", "receiver_stats:rx_amount_sum_total", "receiver_stats:rx_last_step"],
    entity_rows=[{"nameDest": r} for r in demo_receivers],
).to_dict()
pd.DataFrame(online)

Materializing 1 feature views from 2025-12-31 01:00:00+00:00 to 2026-02-01 23:00:00+00:00 into the sqlite online store.

receiver_stats:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,nameDest,rx_txn_count_total,rx_last_step,rx_amount_sum_total
0,C1032986144,17,345,"2,109,076.860"
1,C932583850,51,385,"23,787,558.800"
2,C2094777811,39,503,"26,410,831.150"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**How the API will use this on Day 7-8:** a request arrives with `amount`, `step`, `type`, `nameDest`. The API reads the three receiver values above from Feast, then computes `rx_avg_amount`, `rx_hours_since_last` and `amount_to_rx_avg` with the same formulas used in `build_features`.

## 11. Download your files

In [ ]:
sample.to_parquet("outputs/feast_demo_sample.parquet", index=False)
!zip -qr day3-4_outputs.zip outputs feature_repo/feature_store.yaml feature_repo/features.py
try:
    from google.colab import files
    files.download("day3-4_outputs.zip")
except ImportError:
    print("Not in Colab: the zip is saved next to the notebook.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Day 3-4 checklist
- [ ] All cells ran; the consistency check printed **PASSED**
- [ ] You can explain in your own words why a feature must only use earlier information
- [ ] You can explain why receiver history is in Feast and the transaction fields are not
- [ ] You wrote down which base features differ most between fraud and normal
- [ ] You downloaded `day3-4_outputs.zip`

**Next (Day 5-6):** train XGBoost with class weights on `features_full.parquet`, compare **with and without** the balance features, evaluate with precision-recall AUC, and log every run in MLflow.